# Mississippi Basin

Welcome to the real-world application tutorial! In this guide, we walk through a complete, end-to-end data pipeline using the Mississippi River Basin. You will learn how to obtain external land surface model data, couple it to the river network, and run a full RAPID2 simulation.

This tutorial is provided as an interactive python notebook (`ipynb`), and can be opend and run in your browser by clicking the Binder badge:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/c-h-david/rapid-hub/main?labpath=docs/user-guide/examples/tutorial-mississippi.ipynb)

Before starting, esnure that RAPID2 has been installed in your vistual environment (see the [Quickstart Guide](https://c-h-david.github.io/rapid-hub/user-guide/quick-start-sandbox/)). If running this tutorial in Binder, this setup is done automatically. 

First, import Python packages needed for the tutorial.

In [ ]:
import zipfile
from pathlib import Path
import urllib.request

import rapid2 as r2

We can check the version of the installed RAPID2 version using:

In [ ]:
print(r2.__version__)

## 1. Get Static Parameter Files for the Mississippi River

First, let's create working directories where the inputs and outputs of RAPID2 will be saved.

In [ ]:
WORK_DIR = Path("rapid2_tutorial")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Next, we need to get six static parameter files for the Mississippi Basin. We'll be using prepared files from a Zenodo repository:

[RAPID2 static files for global implementation with MERIT Basins v1.0 and GLDAS v2.0](https://zenodo.org/records/20672740)

This repository includes 61 zipped folders corresponding to the global Pfafstetter Level 2 hydrologic regions. For this tutorial, we'll use the `pfaf_74` files, corresponding to the Mississippi
River Basin. We can download `MERIT_Basins_v1.0_pfaf_74_GLDAS_v2.0_RAPID_v2.0.zip` using `wget`, and then extract the files using `unzip`.

In [ ]:
ZENODO_RECORD = "20672740"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

print(f"Downloading Mississippi ZIP file from Zenodo record {ZENODO_RECORD}...")
url = f"{ZENODO_BASE}/MERIT_Basins_v1.0_pfaf_74_GLDAS_v2.0_RAPID_v2.0.zip?download=1"
dest = INPUT_DIR / 'pfaf_74' 
if not dest.exists():
    urllib.request.urlretrieve(url, dest)
else:
    print(f'\tFiles already downloaded')
with zipfile.ZipFile(dest, "r") as z:
        z.extractall(INPUT_DIR)
print(f"\nStatic files for Mississippi Basin:")
for f in sorted(INPUT_DIR.glob(f"*pfaf_74*")):
    print(f"  {f.name}")

After opening the folder, it contains six `parquet` files needed to run RAPID2. Here is a quick overview of the input files:

1. `bas_pfaf_74_topo.parquet`: List of reaches within the basin.
2. `con_pfaf_74.parquet`: Network topology of the reaches, representing upstream/downstream connectivity.
3. `cpl_pfaf_74_GLDAS.parquet`: Spatial linkage between the land surface model outputs and hydrography network.
4. `crd_pfaf_74.parquet`: Coordinates of each reach polyline.
5. `kpr_pfaf_74_nrm.parquet`: Parameter *k* that represents the flow wave travel time.
6. `xpr_pfaf_74_nrm.parquet`: Dimensionless parameter *x* that relates to flow wave attenuation.

Files 1-4 correspond to the hydrography of the MERIT-Hydro v0.7 Basins v1.0. dataset. Meanwhile, files 5-6 include reach-specific values of *k* and *x*, which are Muskingum routing parameters.

> Note: It is possible to adjust the residence times related to the *k* and *x* parameters. Three different residence times are supported: low (`low`), medium (`nrm`), and high (`high`). These files can also be found [on Zenodo](https://doi.org/10.5281/zenodo.8248069) within zipped folders corresponding to the Pfafstetter Level 2 basins (e.g., `k_pfaf_ii_low.zip`, `x_pfaf_ii_hig.zip`). For more information, please see [Collins et al., 2024. *Nature Geoscience.*](https://doi.org/10.1038/s41561-024-01421-5)


## 2. Download Raw Runoff Data with `dgldas2`

The RAPID2 routing model requires runoff inputs from a land surface model. We'll use runoff data downloaded from the Global Land Data Assimilation System (GLDAS) using the `pull.gldas` function that is part of the `rapid2` package.

A NASA Earthdata account is required to download the GLDAS data. If you have not yet done so, you may also need to accept an end-user license agreement from the NASA Goddard Earth Sciences (GES) Data and Information Services Center (DISC). The `pull.gldas` function handles Earthdata authentication with user input username and password.

For this tutorial, we'll download GLDAS phase `2.1`, model `VIC`, for January, 2010.

In [ ]:
GLDAS_FILE = INPUT_DIR / f'GLDAS_2.1_VIC_2010-01.nc4'
r2.pull.gldas2(
    YS_phs='2.1', 
    YS_mod='VIC', 
    YS_tim='2010-01', 
    lsm_ncf=GLDAS_FILE
    )

A new `GLADS` runoff file is is written to `input/Tutorial`. Other files are also automatically
downloaded in that directory and removed after use.

> Note: GLDAS `2.0` is also supported, as are the `CLSM` and `NOAH` runoff models.

## 3. Couple the Runoff Data with the River Network 

Once you have the raw land surface model data and the static river basin parameters, the two must be linked by mapping the runoff to the river network.. The `prep.couple` function transforms the gridded runoff data into RAPID2's reach-based external inflow format. By providing the land surface model data, connectivity, coordinates, and coupling files, this tool generates the external inflow NetCDF file required for routing.

In [ ]:
QEXT_FILE = INPUT_DIR / f"Qext_pfaf_74_2.1_VIC_2010-01.nc4"
r2.prep.couple(
    lsm_ncf=GLDAS_FILE, 
    con_pqt=f'{INPUT_DIR}/con_pfaf_74.parquet', 
    crd_pqt=f'{INPUT_DIR}/crd_pfaf_74.parquet', 
    cpl_pqt=f'{INPUT_DIR}/cpl_pfaf_74_GLDAS.parquet', 
    Qex_ncf=QEXT_FILE
    )

This creates an external inflow file (`Qext`) in the input directory.

## 4. Create Cold Start Initial Discharge File

At the start of our simulation, we need to provide RAPID2 with initial values of discharge. The `prep.coldinit` function creates a cold start file with all initial discharge set to zero.

In [ ]:
QINIT_FILE = INPUT_DIR / f"Qinit_pfaf_74_2.1_VIC_2010-01.nc4"
r2.prep.coldinit(
    Qex_ncf=QEXT_FILE,
    Q00_ncf=QINIT_FILE
    )

This writes an all zero initial discharge (`Qinit`) file to the input directory.

## 5. Run RAPID2 Routing Model

Finally, a RAPID2 routing model simulation can be run using the `core.run` function, passing as inputs: 
1) Static parameter files 
2) Initital and external flow files 
3) Routing time step in seconds (`IS_dtR`) 
4) Location and names of the output files.

In [ ]:
r2.core.run(
    Qex_ncf = INPUT_DIR / 'Qext_pfaf_74_2.1_VIC_2010-01.nc4',
    Q00_ncf = INPUT_DIR / 'Qinit_pfaf_74_2.1_VIC_2010-01.nc4',
    con_pqt = INPUT_DIR / 'con_pfaf_74.parquet',
    kpr_pqt = INPUT_DIR / 'kpr_pfaf_74_nrm.parquet',
    xpr_pqt = INPUT_DIR / 'xpr_pfaf_74_nrm.parquet',
    bas_pqt = INPUT_DIR / 'bas_pfaf_74_topo.parquet',
    IS_dtR = 900,
    Qou_ncf= OUTPUT_DIR / 'Qout_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4',
    Qfi_ncf= OUTPUT_DIR / 'Qfinal_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4',
)

That's it!

Our output files were written to the `output/Tutorial` folder. `Qout_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4` represents the discharge time series for the Mississippi River Basin in January 2010. Meanwhile, `Qfinal_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4` gives us the final state for a model restart.

> Note: To run RAPID2 for longer than one month, the simulation can be extended by using the Qfinal file from the previous time step (`Qfi_ncf`) as the Qinit file for the next time step (`Q00_ncf`). It is currently possible in RAPID2 to download up to 300 time steps of runoff data from GLDAS at once, which is about 37.5 days of data.